In [3]:
"""
26 Feb 2026
fixed_point_comparison.py
=========================
Compares the three bilingualism attractors of the extended Abrams-Strogatz model:

    x_o†   zero of g alone  — purely demographic baseline
    x_o^f  zero of f alone  — purely linguistic attractor (ASM)
    x_o*   zero of f + g    — attractor of the complete system

KEY PHYSICAL RESULT — sign of Δ_shift = x_o* − x_o^f is REGIME-DEPENDENT
--------------------------------------------------------------------------

  a > 1  (Configuration A, B1):  x_o^f ≈ 0  (UNSTABLE repeller near zero)
       ⟹  x_o† < x_o^f ≪ x_o*     Δ_shift > 0
       The demographic term g LIFTS x_o* above the repeller AND stabilises it.
       Without g the system would collapse to x_o = 0 (full monolingualism).

  a < 1  (Configuration B2):     x_o^f ≈ 0.965  (STABLE attractor near one)
       ⟹  x_o† < x_o* < x_o^f     Δ_shift < 0
       g PULLS x_o* DOWN from the linguistic attractor.
       Demographic dilution (births at finite rate p_o < 1, quotient-rule
       correction) prevents full convergence to x_o^f.
       Census data (0.66–0.89) lies between x_o† and x_o^f, as predicted.

Gap decomposition  (all quantities signed)
------------------------------------------
    Δ_total  = x_o* − x_o†      total shift from demographic baseline
    Δ_demog  = x_o^f − x_o†     pure-prestige gap above demographic baseline  (> 0 always)
    Δ_shift  = x_o* − x_o^f     displacement of ASM FP by g  (sign varies)
    Identity: Δ_total = Δ_demog + Δ_shift  (exact, verified numerically)

"Year of convergence" for B2
------------------------------
    The ODE trajectory from x_o_obs(1970) = 0.7235 rises monotonically toward
    x_o^f ≈ 0.9649 from BELOW (never overshoots: x_o*(t) < x_o^f for all finite t).
    As I → K_I the logistic slows, g → 0, and x_o*(∞) → x_o^f (gap < 1e-4).
    The script reports the year at which |x_o(t) − x_o^f| < ε for several ε.

Author: Riccardo Del Gratta
Date: February 2026
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.optimize import brentq
from scipy.integrate import solve_ivp
import os
import warnings

# ============================================================================
# MODEL PARAMETERS  (identical across all scripts in this repository)
# ============================================================================

N0_indigenous = 1012848
K_indigenous  = 12918933
r_indigenous  = 0.022

N0_spanish = 3117878
K_spanish  = 165995301
r_spanish  = 0.037

A_p   = 5.47
nu    = 0.271
p_max = 0.97

base_year = 1895

# ---------------------------------------------------------------------------
# THREE SCENARIOS
# ---------------------------------------------------------------------------
S_O_CONS_ALL = 0.0349;  S_L_CONS_ALL = 0.0055;  A_CONS_ALL = 1.25    # A
S_O_CONS     = 0.0454;  S_L_CONS     = 0.00839; A_CONS     = 1.478   # B1
S_O_COEX     = 0.047;   S_L_COEX     = 0.015;   A_COEX     = 0.6553  # B2

T_REGIME_CHANGE = 75.0   # t = year 1970

# Census data  (x_o_obs = Bilingual / Indigenous)
CENSUS_XO_OBS = {
    1895: 0.2886, 1900: 0.2949, 1910: 0.2951, 1930: 0.4754,
    1940: 0.5032, 1950: 0.6752, 1960: 0.6353, 1970: 0.7235,
    1980: 0.7591, 1990: 0.8352, 1995: 0.8519, 2000: 0.8310,
    2005: 0.8772, 2010: 0.8479, 2020: 0.8895,
}


# ============================================================================
# CORE FUNCTIONS  (self-contained — no import from sibling scripts)
# ============================================================================

def logistic(t, K, r, N0):
    return K / (1 + ((K - N0) / N0) * np.exp(-r * t))

def p_o_func(m):
    return p_max / (1 + A_p * np.exp(-nu * m))

def f_func(x_o, s_o, s_l, a):
    if np.ndim(x_o) == 0:
        if x_o <= 0 or x_o >= 1:
            return 0.0
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            return s_o * (x_o**a) * (1 - x_o) - s_l * ((1 - x_o)**a) * x_o
    else:
        x = np.asarray(x_o, dtype=float)
        out = np.zeros_like(x)
        mask = (x > 0) & (x < 1)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            out[mask] = (s_o*(x[mask]**a)*(1-x[mask])
                        -s_l*((1-x[mask])**a)*x[mask])
        return out

def g_func(x_o, p, m_val, I):
    bracket = p - x_o - p * (1 - p/p_max) * nu * m_val
    return bracket * r_indigenous * (1 - I/K_indigenous)

def rhs(x_o, I, S, s_o, s_l, a):
    m_val = S/I if I > 0 else np.inf
    p     = p_o_func(m_val)
    return f_func(x_o, s_o, s_l, a) + g_func(x_o, p, m_val, I)

def find_stable_fp(I_val, S_val, s_o, s_l, a, n=4000):
    """Return stable interior FP of f+g, or nan."""
    xs   = np.linspace(0.005, 0.995, n)
    vals = np.array([rhs(x, I_val, S_val, s_o, s_l, a) for x in xs])
    for idx in np.where(np.diff(np.sign(vals)))[0]:
        try:
            xr = brentq(lambda x: rhs(x, I_val, S_val, s_o, s_l, a),
                        xs[idx], xs[idx+1], xtol=1e-10)
            slope = (rhs(xr+1e-6, I_val, S_val, s_o, s_l, a)
                    -rhs(xr-1e-6, I_val, S_val, s_o, s_l, a)) / 2e-6
            if slope < 0:
                return xr
        except Exception:
            pass
    return np.nan

def demographic_fp(I_val, S_val):
    """x_o† = p_o * [1 - (1 - p_o/p_max)*nu*m_si]  — zero of g alone."""
    m  = S_val / I_val
    p  = p_o_func(m)
    xd = p * (1.0 - (1.0 - p/p_max) * nu * m)
    return xd if 0 < xd < 1 else np.nan

def asm_fp(s_o, s_l, a):
    """
    x_o^f = 1/(1+β),  β = (s_o/s_l)^{1/(a-1)}.
    x_o^f = β/(1+β),  β = (s_l/s_o)^{1/(a-1)}.

    Returns (x_o^f, β, is_stable).
    is_stable = True iff a < 1  (f' < 0 at x_o^f → stable attractor of f alone).
    is_stable = False iff a > 1 (f' > 0 at x_o^f → unstable repeller of f alone).
    """
    if abs(a - 1) < 1e-12:
        return np.nan, np.nan, None
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        #beta      = (s_o/s_l) ** (1.0/(a - 1.0))
        beta      = (s_l/s_o) ** (1.0/(a - 1.0))
        print("MOLLY ", s_l, s_o, a)
        xf        = beta / (1.0 + beta)
        is_stable = (a < 1)
    return xf, beta, is_stable


# ============================================================================
# TIME SERIES
# ============================================================================

def compute_triad(s_o, s_l, a, t_arr):
    """Compute x_o†(t), x_o^f (scalar), x_o*(t) over t_arr."""
    xf, beta, is_stable = asm_fp(s_o, s_l, a)
    xd_arr    = np.full(len(t_arr), np.nan)
    xstar_arr = np.full(len(t_arr), np.nan)
    for k, t in enumerate(t_arr):
        I_v = logistic(t, K_indigenous, r_indigenous, N0_indigenous)
        S_v = logistic(t, K_spanish,    r_spanish,    N0_spanish)
        xd_arr[k]    = demographic_fp(I_v, S_v)
        xstar_arr[k] = find_stable_fp(I_v, S_v, s_o, s_l, a)
    return dict(xf=xf, beta=beta, is_stable=is_stable,
                xd=xd_arr, xstar=xstar_arr)


# ============================================================================
# ODE INTEGRATION — B2 trajectory
# ============================================================================

def integrate_ode_B2(t_start=75.0, t_end=600.0, n_pts=30000):
    """
    Integrate B2 ODE from x_o_obs(1970).

    The trajectory rises monotonically from 0.7235 toward x_o^f ≈ 0.9649
    from BELOW.  x_o*(t) < x_o^f for all finite t because g pulls downward.
    As I → K_I, g → 0 and x_o*(∞) → x_o^f (to within ~7e-5).
    """
    x0 = CENSUS_XO_OBS[1970]
    xf, _, _ = asm_fp(S_O_COEX, S_L_COEX, A_COEX)

    def ode(t, y):
        xv = float(np.clip(y[0], 0.001, 0.999))
        I  = logistic(t, K_indigenous, r_indigenous, N0_indigenous)
        S  = logistic(t, K_spanish,    r_spanish,    N0_spanish)
        return [rhs(xv, I, S, S_O_COEX, S_L_COEX, A_COEX)]

    t_eval = np.linspace(t_start, t_end, n_pts)
    sol    = solve_ivp(ode, [t_start, t_end], [x0],
                       t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-11)
    return dict(t_arr=base_year + sol.t, x_arr=sol.y[0], xf=xf)


# ============================================================================
# TABLE 1 — attractor triad at census years
# ============================================================================

def print_attractor_table():
    xf_A,  beta_A,  _ = asm_fp(S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL)
    xf_B1, beta_B1, _ = asm_fp(S_O_CONS,     S_L_CONS,     A_CONS)
    xf_B2, beta_B2, _ = asm_fp(S_O_COEX,     S_L_COEX,     A_COEX)

    W = 155
    print("\n" + "="*W)
    print("  ATTRACTOR TRIAD  at census years")
    print("  x_o† = zero of g (demographic baseline, regime-independent)")
    print("  x_o^f = 1/(1+β), β=(s_o/s_l)^{1/(a-1)}  (ASM pure FP, TIME-INDEPENDENT)")
    print("  x_o*  = stable FP of f+g (complete system)")
    print()
    print("  Gap decomposition (signed):  Δ_tot = Δ_demog + Δ_shift")
    print("    Δ_tot   = x_o* − x_o†          Δ_demog = x_o^f − x_o†")
    print("    Δ_shift = x_o* − x_o^f          [> 0 for A,B1 (g lifts);  < 0 for B2 (g pulls down)]")
    print("="*W)

    hdr = (f"{'Yr':>4}  {'obs':>6}  {'xd':>6}  "
           f"{'xf_A':>7}  {'x*_A':>6}  {'Dtot_A':>7}  {'Dshft_A':>8}  "
           f"||  {'xf_B1':>7}  {'x*_B1':>6}  {'Dtot_B1':>8}  {'Dshft_B1':>9}  "
           f"||  {'xf_B2':>7}  {'x*_B2':>6}  {'Dtot_B2':>8}  "
           f"{'Dshft_B2':>9}  {'Ddemog_B2':>10}  {'check':>7}")
    print(hdr)
    print("-"*W)

    for year in sorted(CENSUS_XO_OBS):
        t   = year - base_year
        I_t = logistic(t, K_indigenous, r_indigenous, N0_indigenous)
        S_t = logistic(t, K_spanish,    r_spanish,    N0_spanish)
        obs = CENSUS_XO_OBS[year]
        xd  = demographic_fp(I_t, S_t)

        # Configuration A
        xs_A     = find_stable_fp(I_t, S_t, S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL)
        dtot_A   = xs_A - xd    if np.isfinite(xs_A) else np.nan
        dshft_A  = xs_A - xf_A  if np.isfinite(xs_A) else np.nan

        # Configuration B1  (pre-1970 only)
        if year <= 1970:
            xs_B1    = find_stable_fp(I_t, S_t, S_O_CONS, S_L_CONS, A_CONS)
            dtot_B1  = xs_B1 - xd    if np.isfinite(xs_B1) else np.nan
            dshft_B1 = xs_B1 - xf_B1 if np.isfinite(xs_B1) else np.nan
        else:
            xs_B1 = dtot_B1 = dshft_B1 = np.nan

        # Configuration B2  (post-1970 only)
        if year >= 1970:
            xs_B2      = find_stable_fp(I_t, S_t, S_O_COEX, S_L_COEX, A_COEX)
            dtot_B2    = xs_B2 - xd    if np.isfinite(xs_B2) else np.nan
            dshft_B2   = xs_B2 - xf_B2 if np.isfinite(xs_B2) else np.nan   # < 0
            ddemog_B2  = xf_B2 - xd    if np.isfinite(xs_B2) else np.nan   # > 0
            check      = ddemog_B2 + dshft_B2 if np.isfinite(xs_B2) else np.nan
        else:
            xs_B2 = dtot_B2 = dshft_B2 = ddemog_B2 = check = np.nan

        def f(v, w=7):
            return f"{v:{w}.4f}" if np.isfinite(v) else f"{'---':>{w}}"

        flag = " ←" if year == 1970 else ""
        print(f"{year:>4}  {obs:>6.4f}  {f(xd,6)}  "
              f"{f(xf_A)}  {f(xs_A,6)}  {f(dtot_A)}  {f(dshft_A,8)}  "
              f"||  {f(xf_B1)}  {f(xs_B1,6)}  {f(dtot_B1,8)}  {f(dshft_B1,9)}  "
              f"||  {f(xf_B2)}  {f(xs_B2,6)}  {f(dtot_B2,8)}  "
              f"{f(dshft_B2,9)}  {f(ddemog_B2,10)}  {f(check,7)}{flag}")

    print()
    print("  LEGEND")
    print("    xd        = x_o† (demographic baseline)")
    print("    xf_X      = x_o^f = 1/(1+β)  (time-independent ASM FP)")
    print("    x*_X      = x_o*  (stable FP of f+g)")
    print("    Dtot      = x_o* − x_o†    (total shift from demographic baseline)")
    print("    Dshft     = x_o* − x_o^f   (how g displaces the ASM FP)")
    print("                > 0 for A, B1 (a>1): g LIFTS x_o* above the unstable repeller")
    print("                < 0 for B2 (a<1):    g PULLS x_o* below the stable attractor")
    print("    Ddemog_B2 = x_o^f − x_o†   (prestige gap above demographic baseline)")
    print("    check     = Ddemog + Dshft  (must equal Dtot exactly)")
    print()
    print("  ORDERING:")
    print("    A, B1 (a>1): x_o† ≈ x_o^f ≪ x_o*   [both dagger and xf are near zero]")
    print("    B2 (a<1):    x_o† < x_o*  < x_o^f  [xf≈0.965, x* lies below in 0.66-0.85]")
    print()


# ============================================================================
# TABLE 2 — convergence analysis (B2)
# ============================================================================

def print_convergence_table():
    xf, _, _ = asm_fp(S_O_COEX, S_L_COEX, A_COEX)

    # x_o*(∞): FP of f+g at I → K_I, S → K_S
    xstar_inf = find_stable_fp(K_indigenous*0.9999, K_spanish*0.9999,
                               S_O_COEX, S_L_COEX, A_COEX)

    result = integrate_ode_B2(t_start=75.0, t_end=600.0, n_pts=50000)
    t_arr  = result['t_arr']
    x_arr  = result['x_arr']

    print("\n" + "="*72)
    print("  CONVERGENCE — Configuration B2  (a=0.655)")
    print(f"  Initial condition: x_o(1970) = {CENSUS_XO_OBS[1970]:.4f}  (census)")
    print(f"  x_o^f      = {xf:.5f}  (ASM FP, STABLE for a<1, approached from below)")
    print(f"  x_o*(inf)  = {xstar_inf:.5f}  (f+g FP as I→K_I; gap |x_o^f - x_o*(inf)| ≈ {abs(xf-xstar_inf):.1e})")
    print()
    print("  The trajectory rises MONOTONICALLY from 0.7235 toward x_o^f from below.")
    print("  x_o*(t) < x_o^f for all finite t: g continuously pulls the attractor")
    print("  downward; as I→K_I the logistic term decays and g→0, so x_o*(∞)→x_o^f.")
    print()
    print(f"  {'epsilon':>8}  {'year: x_o reaches x_o^f-ε':>26}  {'x_o at that year':>17}")
    print("  " + "-"*55)

    for eps in (0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.001):
        idx = np.where(np.abs(x_arr - xf) < eps)[0]
        if len(idx):
            yr = t_arr[idx[0]]
            xv = x_arr[idx[0]]
            print(f"  {eps:>8.3f}  {yr:>26.1f}  {xv:>17.5f}")
        else:
            print(f"  {eps:>8.3f}  {'> 2495':>26}  {'---':>17}")

    print()
    print("  Sample trajectory values:")
    for yr_t in (2020, 2050, 2100, 2150, 2200, 2300, 2400):
        idx = np.argmin(np.abs(t_arr - yr_t))
        print(f"    year {yr_t}: x_o = {x_arr[idx]:.5f}   gap to x_o^f: {xf - x_arr[idx]:.5f}")
    print()


# ============================================================================
# FIGURE 1 — time-series triad  (3 panels)
# ============================================================================

def plot_attractor_triad(t_start=0.0, t_end=130.0, n_t=400,
                         plot_dir='plots/fp_comparison/'):
    """
    Panel 1: x_o†(t), x_o^f (horizontal lines), x_o*(t)  for A, B1, B2.
    Panel 2: gap decomposition Δ_total, Δ_shift, Δ_demog  for A and B2.
    Panel 3: B2 ODE trajectory from census 1970 converging to x_o^f from below.
    """
    t_arr    = np.linspace(t_start, t_end, n_t)
    year_arr = base_year + t_arr
    mask_pre  = t_arr < T_REGIME_CHANGE
    mask_post = t_arr >= T_REGIME_CHANGE

    print("    Computing A  triad ...", flush=True)
    tr_A  = compute_triad(S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL, t_arr)
    print("    Computing B1 triad ...", flush=True)
    tr_B1 = compute_triad(S_O_CONS,     S_L_CONS,     A_CONS,     t_arr[mask_pre])
    print("    Computing B2 triad ...", flush=True)
    tr_B2 = compute_triad(S_O_COEX,     S_L_COEX,     A_COEX,     t_arr[mask_post])
    print("    Integrating B2 ODE  ...", flush=True)
    ode   = integrate_ode_B2(t_start=75.0, t_end=450.0, n_pts=20000)

    xf_A,  _, _ = asm_fp(S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL)
    xf_B1, _, _ = asm_fp(S_O_CONS,     S_L_CONS,     A_CONS)
    xf_B2, _, _ = asm_fp(S_O_COEX,     S_L_COEX,     A_COEX)

    xd_arr  = tr_A['xd']         # shared demographic baseline
    xd_post = tr_B2['xd']

    obs_yrs = np.array(sorted(CENSUS_XO_OBS))
    obs_val = np.array([CENSUS_XO_OBS[y] for y in obs_yrs])

    # ── layout ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(15, 17))
    gs  = gridspec.GridSpec(3, 1, hspace=0.58)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])

    for ax in (ax1, ax2):
        ax.axvspan(base_year, 1970,          alpha=0.06, color='steelblue', zorder=0)
        ax.axvspan(1970, year_arr[-1]+2,     alpha=0.06, color='tomato',    zorder=0)
        ax.axvline(1970, color='grey', lw=0.9, ls='--', zorder=1)
        ax.set_xlim(year_arr[0]-2, year_arr[-1]+2)

    # ── Panel 1 ───────────────────────────────────────────────────────────
    # x_o† (demographic baseline, shared)
    ax1.plot(year_arr, xd_arr, color='mediumpurple', lw=2.2, zorder=4,
             label=r'$x_o^\dagger(t)$ — demographic baseline (zero of $g$ alone)')

    # x_o^f horizontal lines
    ax1.axhline(xf_A,  color='steelblue', lw=1.5, ls='--', alpha=0.7,
                label=fr'$x_o^f$ A  ($a=1.25$,  UNSTABLE) $= {xf_A:.4f}$')
    ax1.axhline(xf_B1, color='tomato',    lw=1.5, ls=':',  alpha=0.7,
                label=fr'$x_o^f$ B1 ($a=1.478$, UNSTABLE) $= {xf_B1:.4f}$')
    ax1.axhline(xf_B2, color='darkred',   lw=1.8, ls='-.', alpha=0.85,
                label=fr'$x_o^f$ B2 ($a=0.655$, STABLE)   $= {xf_B2:.4f}$')

    # x_o*(t)
    ax1.plot(year_arr, tr_A['xstar'],
             color='steelblue', lw=2.2, ls='-', zorder=5,
             label=r'$x_o^*(t)$ — A  (f+g)')
    ax1.plot(year_arr[mask_pre],  tr_B1['xstar'],
             color='tomato', lw=2.0, ls='--', zorder=5,
             label=r'$x_o^*(t)$ — B1 (1895-1970)')
    ax1.plot(year_arr[mask_post], tr_B2['xstar'],
             color='tomato', lw=2.0, ls='-.', zorder=5,
             label=r'$x_o^*(t)$ — B2 (1970-2020)')

    # Census
    ax1.scatter(obs_yrs[obs_yrs<=1970], obs_val[obs_yrs<=1970],
                color='steelblue', marker='s', s=55, zorder=7,
                edgecolors='navy', lw=0.8, label=r'census $x_o^{\rm obs}$ (pre-1970)')
    ax1.scatter(obs_yrs[obs_yrs>=1970], obs_val[obs_yrs>=1970],
                color='tomato',    marker='s', s=55, zorder=7,
                edgecolors='darkred', lw=0.8, label=r'census $x_o^{\rm obs}$ (post-1970)')

    # Annotations
    ax1.annotate(r'$x_o^f$(B2)$\approx 0.965$: $g$ PULLS $x_o^*$ DOWN'
                 '\n' r'($\Delta_{\rm shift}<0$ for $a<1$)',
                 xy=(2010, xf_B2), xytext=(1960, 0.90),
                 fontsize=8.5, color='darkred',
                 arrowprops=dict(arrowstyle='->', color='darkred', lw=0.9),
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='darkred', alpha=0.9))
    ax1.annotate(r'$x_o^f$(A)$\approx 0$: $g$ LIFTS $x_o^*$ UP'
                 '\n' r'($\Delta_{\rm shift}>0$ for $a>1$)',
                 xy=(1920, xf_A+0.005), xytext=(1907, 0.14),
                 fontsize=8.5, color='steelblue',
                 arrowprops=dict(arrowstyle='->', color='steelblue', lw=0.9),
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='steelblue', alpha=0.9))

    ax1.set_ylabel(r'$x_o$', fontsize=13)
    ax1.set_ylim(-0.02, 1.02)
    ax1.grid(alpha=0.25)
    ax1.set_title(
        r'Attractor triad: $x_o^\dagger(t)$ — $x_o^f$ (horizontal lines) — $x_o^*(t)$'
        '\n'
        r'A/B1 ($a>1$): $x_o^f\approx0$ (unstable), $x_o^\dagger\!\approx\!x_o^f\!\ll\!x_o^*$ — $g$ lifts and stabilises'
        '\n'
        r'B2 ($a<1$): $x_o^f\approx0.965$ (stable), $x_o^\dagger\!<\!x_o^*\!<\!x_o^f$ — $g$ pulls down from linguistic attractor',
        fontsize=10.5)
    ax1.legend(fontsize=8.2, loc='upper center',
               bbox_to_anchor=(0.5, -0.17), ncol=3, borderaxespad=0, framealpha=0.95)

    # ── Panel 2 — gap decomposition ───────────────────────────────────────
    d_tot_A   = tr_A['xstar'] - xd_arr
    d_shft_A  = tr_A['xstar'] - xf_A             # > 0 throughout

    d_tot_B2  = tr_B2['xstar'] - xd_post
    d_shft_B2 = tr_B2['xstar'] - xf_B2           # < 0 throughout
    d_demog_B2 = xf_B2 - xd_post                 # > 0 (scalar - array)

    ax2.axhline(0, color='k', lw=0.8)
    ax2.plot(year_arr, d_tot_A,
             color='steelblue', lw=2.0, ls='-',
             label=r'$\Delta_{\rm tot}=x_o^*-x_o^\dagger$ — A')
    ax2.plot(year_arr, d_shft_A,
             color='steelblue', lw=1.6, ls='--',
             label=r'$\Delta_{\rm shift}=x_o^*-x_o^f>0$ — A (g lifts)')
    ax2.plot(year_arr[mask_post], d_tot_B2,
             color='tomato', lw=2.0, ls='-.',
             label=r'$\Delta_{\rm tot}=x_o^*-x_o^\dagger$ — B2')
    ax2.plot(year_arr[mask_post], d_shft_B2,
             color='tomato', lw=1.6, ls=':',
             label=r'$\Delta_{\rm shift}=x_o^*-x_o^f<0$ — B2 (g pulls down)')
    ax2.plot(year_arr[mask_post], d_demog_B2,
             color='mediumpurple', lw=1.8, ls='--',
             label=r'$\Delta_{\rm demog}=x_o^f-x_o^\dagger$ — B2 (pure prestige gap, $>0$)')

    ax2.fill_between(year_arr[mask_post], d_shft_B2, 0,
                     alpha=0.13, color='tomato')
    ax2.fill_between(year_arr, d_shft_A, 0,
                     alpha=0.08, color='steelblue')

    ax2.set_ylabel(r'Gap $\Delta$  (signed)', fontsize=12)
    ax2.set_ylim(-0.38, 0.75)
    ax2.grid(alpha=0.25)
    ax2.set_title(
        r'Gap decomposition (signed): $\Delta_{\rm tot} = \Delta_{\rm demog} + \Delta_{\rm shift}$'
        '\n'
        r'$\Delta_{\rm shift}>0$ (A): g lifts $x_o^*$ above unstable repeller $x_o^f\approx0$'
        r'  ·  '
        r'$\Delta_{\rm shift}<0$ (B2): g pulls $x_o^*$ below stable attractor $x_o^f\approx0.965$',
        fontsize=10.5)
    ax2.legend(fontsize=8.2, loc='upper center',
               bbox_to_anchor=(0.5, -0.20), ncol=3, borderaxespad=0, framealpha=0.95)

    # ── Panel 3 — B2 ODE trajectory ───────────────────────────────────────
    t_traj = ode['t_arr']
    x_traj = ode['x_arr']
    xf_v   = ode['xf']

    mask_d = t_traj <= 2450
    ax3.plot(t_traj[mask_d], x_traj[mask_d],
             color='tomato', lw=2.2,
             label=r'B2 trajectory $x_o(t)$ from $x_o^{\rm obs}(1970)=0.7235$')
    ax3.axhline(xf_v, color='darkred', lw=1.8, ls='-.', alpha=0.85,
                label=fr'$x_o^f = {xf_v:.5f}$  (stable ASM FP, approached from below)')

    # Convergence bands (ε = 0.01 and 0.05)
    for eps, alpha in [(0.05, 0.12), (0.01, 0.18)]:
        ax3.fill_between([1968, 2455], xf_v-eps, xf_v,
                         alpha=alpha, color='darkred',
                         label=fr'$\epsilon={eps}$ convergence band')
        idx = np.where(np.abs(x_traj - xf_v) < eps)[0]
        if len(idx) and t_traj[idx[0]] <= 2450:
            yr = t_traj[idx[0]]
            ax3.axvline(yr, color='darkred', lw=1.0, ls=':', alpha=0.7)
            ax3.scatter(yr, x_traj[idx[0]], color='darkred', marker='*', s=160,
                        edgecolors='k', lw=0.7, zorder=6,
                        label=fr'$|x_o-x_o^f|<{eps}$ at year {yr:.0f}')

    # Census markers for B2 period
    for yr_c in [1970, 1980, 1990, 2000, 2010, 2020]:
        if yr_c in CENSUS_XO_OBS:
            ax3.scatter(yr_c, CENSUS_XO_OBS[yr_c], color='k', marker='s', s=45,
                        zorder=7, edgecolors='dimgray', lw=0.6)
    ax3.scatter([], [], color='k', marker='s', s=45, label=r'census $x_o^{\rm obs}$')

    ax3.annotate(
        r'$x_o^*(t) < x_o^f$ for all finite $t$' '\n'
        r'$g$ pulls attractor below linguistic FP' '\n'
        r'Convergence is asymptotic as $I\to K_I$',
        xy=(2280, 0.945), xytext=(2100, 0.800),
        fontsize=8.5, color='darkred',
        arrowprops=dict(arrowstyle='->', color='darkred', lw=0.9),
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='darkred', alpha=0.9))

    ax3.set_xlabel('Year', fontsize=12)
    ax3.set_ylabel(r'$x_o$', fontsize=13)
    ax3.set_xlim(1968, 2455)
    ax3.set_ylim(0.69, 0.982)
    ax3.axvspan(1968, 1972, alpha=0.12, color='steelblue', zorder=0)
    ax3.grid(alpha=0.25)
    ax3.set_title(
        r'B2: ODE trajectory converging monotonically to $x_o^f$ from below'
        '\n'
        r'$x_o^*(t)<x_o^f$ at all times; convergence is asymptotic ($g\to0$ as $I\to K_I$)',
        fontsize=10.5)
    ax3.legend(fontsize=8.2, loc='upper center',
               bbox_to_anchor=(0.5, -0.22), ncol=2, borderaxespad=0, framealpha=0.95)

    fig.suptitle(
        r'Fixed-point comparison: $x_o^\dagger(t)$  vs  $x_o^f$  vs  $x_o^*(t)$  — Configurations A, B1, B2'
        '\n'
        r'$g$ LIFTS $x_o^*$ above $x_o^f$ for $a>1$ ($\Delta_{\rm shift}>0$)  ·  '
        r'$g$ PULLS $x_o^*$ below $x_o^f$ for $a<1$ ($\Delta_{\rm shift}<0$)',
        fontsize=12, fontweight='bold', y=1.005)

    os.makedirs(plot_dir, exist_ok=True)
    fname = f"{plot_dir}attractor_triad.jpg"
    fig.savefig(fname, dpi=300, bbox_inches='tight')
    print(f"    Saved: {fname}")
    plt.close(fig)


# ============================================================================
# FIGURE 2 — horizontal bar chart at census years
# ============================================================================

def plot_fp_bars(plot_dir='plots/fp_comparison/'):
    """
    3 panels (A, B1, B2).  Each census year: stacked horizontal bar
      purple = Δ_demog = x_o^f − x_o†   (always positive)
      blue/red = Δ_shift = x_o* − x_o^f  (positive for A, B1; NEGATIVE for B2 → orange)
    Makes the sign reversal of Δ_shift immediately visible.
    """
    census_years = sorted(CENSUS_XO_OBS)
    xf_A,  _, _ = asm_fp(S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL)
    xf_B1, _, _ = asm_fp(S_O_CONS,     S_L_CONS,     A_CONS)
    xf_B2, _, _ = asm_fp(S_O_COEX,     S_L_COEX,     A_COEX)

    fig, axes = plt.subplots(1, 3, figsize=(17, 9), sharey=True)
    specs = [
        (census_years,                        xf_A,
         S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL, 'steelblue',
         r'A  ($a=1.25$, 1895-2020)',    False),
        ([y for y in census_years if y<=1970], xf_B1,
         S_O_CONS,     S_L_CONS,     A_CONS,     'tomato',
         r'B1 ($a=1.478$, 1895-1970)',  False),
        ([y for y in census_years if y>=1970], xf_B2,
         S_O_COEX,     S_L_COEX,     A_COEX,     'tomato',
         r'B2 ($a=0.655$, 1970-2020)',  True),
    ]

    for ax, (yrs, xf_sc, s_o, s_l, a, clr, ttl, b2) in zip(axes, specs):
        for year in yrs:
            t   = year - base_year
            I_v = logistic(t, K_indigenous, r_indigenous, N0_indigenous)
            S_v = logistic(t, K_spanish,    r_spanish,    N0_spanish)
            xd  = demographic_fp(I_v, S_v)
            xs  = find_stable_fp(I_v, S_v, s_o, s_l, a)
            obs = CENSUS_XO_OBS[year]
            if not np.isfinite(xs):
                continue

            d_demog = xf_sc - xd        # > 0 always
            d_shift = xs    - xf_sc     # > 0 for A/B1; < 0 for B2

            # purple bar: xd → xf
            ax.barh(year, d_demog, left=xd, height=2.5,
                    color='mediumpurple', alpha=0.45, edgecolor='k', lw=0.4)
            # coloured bar: xf → x*  (orange if negative)
            ax.barh(year, d_shift, left=xf_sc, height=2.5,
                    color=('darkorange' if d_shift < 0 else clr),
                    alpha=0.55, edgecolor='k', lw=0.4)

            ax.scatter(xd,    year, color='mediumpurple', marker='D', s=55,
                       zorder=5, edgecolors='indigo', lw=0.8)
            ax.scatter(xf_sc, year, color=clr, marker='o', s=55,
                       zorder=5, edgecolors='k', lw=0.8)
            ax.scatter(xs,    year, color=clr, marker='^', s=70,
                       zorder=5, edgecolors='k', lw=0.8)
            ax.scatter(obs,   year, color='k', marker='s', s=40,
                       zorder=6, edgecolors='dimgray', lw=0.6)

        ax.axvline(xf_sc, color=clr, lw=1.2, ls='--', alpha=0.55)
        ax.set_xlabel(r'$x_o$', fontsize=12)
        ax.set_title(f'Configuration {ttl}', fontsize=11, fontweight='bold')
        ax.set_xlim(0, 1)
        ax.grid(axis='x', alpha=0.25)
        ax.invert_yaxis()

        stab = 'STABLE of $f$' if b2 else 'UNSTABLE repeller of $f$'
        leg_handles = [
            Patch(facecolor='mediumpurple', alpha=0.45,
                  label=r'$\Delta_{\rm demog}=x_o^f-x_o^\dagger>0$'),
            Patch(facecolor=clr,            alpha=0.55,
                  label=r'$\Delta_{\rm shift}=x_o^*-x_o^f>0$ (lifts)'),
            Patch(facecolor='darkorange',   alpha=0.55,
                  label=r'$\Delta_{\rm shift}=x_o^*-x_o^f<0$ (pulls down)'),
            Line2D([0],[0], marker='D', color='mediumpurple', ms=7, ls='none',
                   label=r'$x_o^\dagger$'),
            Line2D([0],[0], marker='o', color=clr, ms=7, ls='none',
                   label=fr'$x_o^f={xf_sc:.4f}$ ({stab})'),
            Line2D([0],[0], marker='^', color=clr, ms=7, ls='none',
                   label=r'$x_o^*$'),
            Line2D([0],[0], marker='s', color='k', ms=6, ls='none',
                   label=r'$x_o^{\rm obs}$ (census)'),
        ]
        ax.legend(handles=leg_handles, fontsize=7.8,
                  loc='upper center', bbox_to_anchor=(0.5, -0.10),
                  ncol=2, borderaxespad=0, framealpha=0.95)

    axes[0].set_ylabel('Year', fontsize=12)
    fig.suptitle(
        r'Attractor gaps at census years: $\Delta_{\rm demog}$ (purple) + $\Delta_{\rm shift}$ (colour / orange)'
        '\n'
        r'Orange bars ($\Delta_{\rm shift}<0$, B2 only) show $g$ REDUCING bilingualism below $x_o^f$',
        fontsize=12, fontweight='bold')
    plt.tight_layout()

    os.makedirs(plot_dir, exist_ok=True)
    fname = f"{plot_dir}attractor_gaps_bars.jpg"
    fig.savefig(fname, dpi=300, bbox_inches='tight')
    print(f"    Saved: {fname}")
    plt.close(fig)


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":

    plot_dir = 'plots/fp_comparison/'
    os.makedirs(plot_dir, exist_ok=True)

    print("\n" + "#"*72)
    print("# fixed_point_comparison.py")
    print("# Attractor triad: x_o† vs x_o^f vs x_o*")
    print("# Configurations A, B1, B2")
    print("#"*72)

    print("\n  SCALAR x_o^f = 1/(1+β), β=(s_o/s_l)^{1/(a-1)}")
    print("  " + "-"*68)
    for sc, s_o, s_l, a in [('A',  S_O_CONS_ALL, S_L_CONS_ALL, A_CONS_ALL),
                              ('B1', S_O_CONS,     S_L_CONS,     A_CONS),
                              ('B2', S_O_COEX,     S_L_COEX,     A_COEX)]:
        xf, beta, is_stable = asm_fp(s_o, s_l, a)
        stab = "STABLE   attractor of f (a<1)" if is_stable \
               else "UNSTABLE repeller of f (a>1)"
        sign = "Δ_shift < 0 — g PULLS DOWN" if is_stable \
               else "Δ_shift > 0 — g LIFTS UP"
        print(f"    {sc}: a={a}  β={beta:.5f}  x_o^f={xf:.5f}  [{stab}]  =>  {sign}")

    print_attractor_table()
    print_convergence_table()

    print("\n[1/2] Plotting attractor triad time series...")
    plot_attractor_triad(plot_dir=plot_dir)

    print("\n[2/2] Plotting attractor gap bars...")
    plot_fp_bars(plot_dir=plot_dir)

    print("\n" + "#"*72)
    print("# KEY FINDINGS")
    print("#")
    print("# 1. x_o^f is TIME-INDEPENDENT (depends only on s_o, s_l, a):")
    print("#      A:  x_o^f ≈ 0.0006  (unstable repeller, β >> 1)")
    print("#      B1: x_o^f ≈ 0.0284  (unstable repeller, β >> 1)")
    print("#      B2: x_o^f ≈ 0.9649  (stable attractor,  β << 1)")
    print("#")
    print("# 2. Δ_shift = x_o* − x_o^f  changes sign between regimes:")
    print("#      a>1 (A, B1): Δ_shift > 0 — g LIFTS x_o* above the repeller")
    print("#                   and STABILISES it (f alone is unstable at x_o^f)")
    print("#      a<1 (B2):    Δ_shift < 0 — g PULLS x_o* below the stable FP")
    print("#                   demographic dilution prevents full bilingualism")
    print("#")
    print("# 3. Δ_total = Δ_demog + Δ_shift  (exact signed identity):")
    print("#      Δ_demog = x_o^f − x_o†  (always > 0: prestige > demographics)")
    print("#      Δ_shift = x_o* − x_o^f  (sign depends on a)")
    print("#")
    print("# 4. B2 ordering: x_o†(t) < x_o*(t) < x_o^f for all 1970-2020")
    print("#    Census data 0.66-0.89 lies between x_o† and x_o^f as expected.")
    print("#    ODE trajectory converges monotonically to x_o^f from below;")
    print("#    x_o^f reached within ε=0.05 around year 2141.")
    print("#"*72)


########################################################################
# fixed_point_comparison.py
# Attractor triad: x_o† vs x_o^f vs x_o*
# Configurations A, B1, B2
########################################################################

  SCALAR x_o^f = 1/(1+β), β=(s_o/s_l)^{1/(a-1)}
  --------------------------------------------------------------------
MOLLY  0.0055 0.0349 1.25
    A: a=1.25  β=0.00062  x_o^f=0.00062  [UNSTABLE repeller of f (a>1)]  =>  Δ_shift > 0 — g LIFTS UP
MOLLY  0.00839 0.0454 1.478
    B1: a=1.478  β=0.02924  x_o^f=0.02841  [UNSTABLE repeller of f (a>1)]  =>  Δ_shift > 0 — g LIFTS UP
MOLLY  0.015 0.047 0.6553
    B2: a=0.6553  β=27.47588  x_o^f=0.96488  [STABLE   attractor of f (a<1)]  =>  Δ_shift < 0 — g PULLS DOWN
MOLLY  0.0055 0.0349 1.25
MOLLY  0.00839 0.0454 1.478
MOLLY  0.015 0.047 0.6553

  ATTRACTOR TRIAD  at census years
  x_o† = zero of g (demographic baseline, regime-independent)
  x_o^f = 1/(1+β), β=(s_o/s_l)^{1/(a-1)}  (ASM pure FP, TIME-IND